# Bullwhip + incoherence example (matched)

This version is aligned with the standalone `bullwhip_CV_incoherence` and `incoherence` notebooks, but uses more robust file loading.

In [1]:

import os
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm


def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("None of these files exists:\n" + "\n".join(map(str, paths)))


def build_candidates(*stems, prefixes=("721", ""), extra_dirs=(".",)):
    paths = []
    for stem in stems:
        for d in extra_dirs:
            for prefix in prefixes:
                if prefix:
                    paths.append(os.path.join(d, f"{prefix}{stem}"))
                paths.append(os.path.join(d, stem))
    return list(dict.fromkeys(paths))


In [2]:

# ---------- configuration ----------
MODELS_DIR = r"E:/yjz/Extension for hts/JayCode/Models/"
PREFIXES = ("721", "")
LEAD_TIME = 1

result_dirs = (".", MODELS_DIR)

lgb_path = first_existing(build_candidates(f"lgbInvtSim_L{LEAD_TIME}.pkl", prefixes=PREFIXES, extra_dirs=result_dirs))
ets_path = first_existing(build_candidates(f"etsInvtSim_L{LEAD_TIME}.pkl", prefixes=PREFIXES, extra_dirs=result_dirs))
bu_path  = first_existing(build_candidates(f"BUOrder_L{LEAD_TIME}.pkl", prefixes=PREFIXES, extra_dirs=result_dirs))
tdfp_path = first_existing(build_candidates(f"TDFPOrder_L{LEAD_TIME}.pkl", prefixes=PREFIXES, extra_dirs=result_dirs))
mint_path = first_existing(build_candidates(
    f"VarOrder_L{LEAD_TIME}.pkl",
    f"MinTOrder_shrink_L{LEAD_TIME}.pkl",
    prefixes=PREFIXES,
    extra_dirs=result_dirs,
))

future_path = first_existing(build_candidates("future_28.pkl", prefixes=PREFIXES, extra_dirs=(MODELS_DIR, ".")))
tag_path = first_existing(build_candidates("tags.bin", "tags.pkl", prefixes=("",), extra_dirs=(MODELS_DIR, ".")))
actual_order_path = first_existing(["actual_order.pkl"])

lgb = pd.read_pickle(lgb_path)
ets = pd.read_pickle(ets_path)
base = pd.concat([lgb, ets], ignore_index=True)
bu = pd.read_pickle(bu_path)
tdfp = pd.read_pickle(tdfp_path)
mint = pd.read_pickle(mint_path)

uid_all = pd.read_pickle(future_path).reset_index(drop=True)
uid = uid_all[["unique_id", "ds"]]
tag = pd.read_pickle(tag_path)
ao = pd.read_pickle(actual_order_path)

print("Loaded:")
for x in [lgb_path, ets_path, bu_path, tdfp_path, mint_path, future_path, tag_path, actual_order_path]:
    print(" -", x)


Loaded:
 - .\lgbInvtSim_L1.pkl
 - .\etsInvtSim_L1.pkl
 - .\BUOrder_L1.pkl
 - .\TDFPOrder_L1.pkl
 - .\VarOrder_L1.pkl
 - E:/yjz/Extension for hts/JayCode/Models/721future_28.pkl
 - E:/yjz/Extension for hts/JayCode/Models/tags.bin
 - actual_order.pkl


## orders_collector

In [3]:

orders = base[["name", "true_demand", "forecasts"]].copy()
for tsl in tqdm(["90", "95", "99"]):
    orders[f"ot_{tsl}"] = base[f"ot_{tsl}"].to_numpy()
    orders[f"ot_bu{tsl}"] = bu[f"ot_{tsl}"].to_numpy()
    orders[f"ot_tdfp{tsl}"] = tdfp[f"ot_{tsl}"].to_numpy()
    orders[f"ot_mint{tsl}"] = mint[f"ot_{tsl}"].to_numpy()

orders.to_pickle("all_order.pkl")
print("saved:", os.path.abspath("all_order.pkl"))
orders.head()


100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 10.55it/s]


saved: E:\yjz\Extension for hts\JayCode\modified_inventory_files\all_order.pkl


,name,true_demand,forecasts,ot_90,ot_bu90,ot_tdfp90,ot_mint90,ot_95,ot_bu95,ot_tdfp95,ot_mint95,ot_99,ot_bu99,ot_tdfp99,ot_mint99
0,lgb_base,4.0,5.689794,5.648312,15.747402,5.648312,10.467349,7.249530,20.211572,7.249530,13.434492,10.253148,28.585611,10.253148,18.999925
1,lgb_base,5.0,4.679130,2.989336,4.350416,2.989336,3.557998,2.989336,4.350416,2.989336,3.557964,2.989336,4.350416,2.989336,3.557878
2,lgb_base,7.0,4.798928,5.119798,4.451770,5.119798,4.807548,5.119798,4.451770,5.119798,4.807558,5.119798,4.451770,5.119798,4.807583
3,lgb_base,1.0,5.980217,8.181289,7.910317,8.181289,7.931240,8.181289,7.910317,8.181289,7.931244,8.181289,7.910317,8.181289,7.931255
4,lgb_base,9.0,5.048564,0.068346,1.017425,0.068346,0.783484,0.068346,1.017425,0.068346,0.783501,0.068346,1.017425,0.068346,0.783545


## products_list

In [4]:

if isinstance(tag, dict):
    if "total/cat_id/dept_id/item_id" in tag:
        series_ids = list(tag["total/cat_id/dept_id/item_id"])
    else:
        raise KeyError("Cannot find key 'total/cat_id/dept_id/item_id' in tag dictionary.")
else:
    # fallback for DataFrame/Series-like tag files
    if "total/cat_id/dept_id/item_id" in getattr(tag, "columns", []):
        series_ids = tag["total/cat_id/dept_id/item_id"].tolist()
    else:
        raise KeyError("Cannot find 'total/cat_id/dept_id/item_id' in tag object.")

prdd = pd.Series(series_ids, name="products").astype(str).str.split("/")
prd = [parts[-1] for parts in prdd]
prd_idx = pd.DataFrame({
    p: uid_all[uid_all["unique_id"].astype(str).str.contains(fr"{p}$", regex=True)].index.tolist()
    for p in tqdm(prd)
})
prd_idx.to_pickle("prd_idx.pkl")
print("saved:", os.path.abspath("prd_idx.pkl"))
prd_idx.iloc[:5, :5]


100%|██████████████████████████████████████████████████████████████████████████████| 3049/3049 [25:58<00:00,  1.96it/s]


saved: E:\yjz\Extension for hts\JayCode\modified_inventory_files\prd_idx.pkl


,FOODS_1_001,FOODS_1_002,FOODS_1_003,FOODS_1_004,FOODS_1_005
0,0,28,56,84,112
1,1,29,57,85,113
2,2,30,58,86,114
3,3,31,59,87,115
4,4,32,60,88,116


## CV and Bullwhip Effect Ratio

In [5]:
orders = pd.read_pickle("all_order.pkl")
uid = uid_all[["unique_id", "ds"]].copy()

lst = orders["name"].unique().tolist()
frdr_ = pd.concat(
    [pd.concat([uid.reset_index(drop=True), orders[orders["name"] == name].reset_index(drop=True)], axis=1) for name in tqdm(lst)],
    ignore_index=True,
)

dff = frdr_.copy()
split_cols = dff["unique_id"].astype(str).str.split("/")
dff["product"] = [parts[-1] for parts in split_cols]
dff["level"] = [len(parts) - 3 for parts in split_cols]
dff.head()


100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:07<00:00,  1.09it/s]


,unique_id,ds,name,true_demand,forecasts,ot_90,ot_bu90,ot_tdfp90,ot_mint90,ot_95,ot_bu95,ot_tdfp95,ot_mint95,ot_99,ot_bu99,ot_tdfp99,ot_mint99,product,level
0,TOTAL/FOODS/FOODS_1/FOODS_1_001,1914,lgb_base,4.0,5.689794,5.648312,15.747402,5.648312,10.467349,7.249530,20.211572,7.249530,13.434492,10.253148,28.585611,10.253148,18.999925,FOODS_1_001,1
1,TOTAL/FOODS/FOODS_1/FOODS_1_001,1915,lgb_base,5.0,4.679130,2.989336,4.350416,2.989336,3.557998,2.989336,4.350416,2.989336,3.557964,2.989336,4.350416,2.989336,3.557878,FOODS_1_001,1
2,TOTAL/FOODS/FOODS_1/FOODS_1_001,1916,lgb_base,7.0,4.798928,5.119798,4.451770,5.119798,4.807548,5.119798,4.451770,5.119798,4.807558,5.119798,4.451770,5.119798,4.807583,FOODS_1_001,1
3,TOTAL/FOODS/FOODS_1/FOODS_1_001,1917,lgb_base,1.0,5.980217,8.181289,7.910317,8.181289,7.931240,8.181289,7.910317,8.181289,7.931244,8.181289,7.910317,8.181289,7.931255,FOODS_1_001,1
4,TOTAL/FOODS/FOODS_1/FOODS_1_001,1918,lgb_base,9.0,5.048564,0.068346,1.017425,0.068346,0.783484,0.068346,1.017425,0.068346,0.783501,0.068346,1.017425,0.068346,0.783545,FOODS_1_001,1


In [6]:
var_ = dff.groupby(["unique_id", "name", "product", "level"]).var(numeric_only=True).drop(columns=["forecasts"], errors="ignore")
metric_cols = [c for c in var_.columns if c != "true_demand"]
var_[metric_cols] = var_[metric_cols].div(var_["true_demand"], axis=0)
var_.head()


ds  \
unique_id                               name     product     level              
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3      35.062371   
                                        ets_bu   FOODS_1_001 3      35.062371   
                                        ets_mint FOODS_1_001 3      35.062371   
                                        ets_td   FOODS_1_001 3      35.062371   
                                        lgb_base FOODS_1_001 3      35.062371   

                                                                    true_demand  \
unique_id                               name     product     level                
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3         1.929894   
                                        ets_bu   FOODS_1_001 3         1.929894   
                                        ets_mint FOODS_1_001 3         1.929894   
                                        ets_td   FOODS_1_001 3         1.929894   
                                        lgb_base FOODS_1_001 3         1.929894   

                                                                       ot_90  \
unique_id                               name     product     level             
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3      1.034852   
                                        ets_bu   FOODS_1_001 3      1.034852   
                                        ets_mint FOODS_1_001 3      1.020277   
                                        ets_td   FOODS_1_001 3      1.031719   
                                        lgb_base FOODS_1_001 3      1.269884   

                                                                     ot_bu90  \
unique_id                               name     product     level             
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3      1.034852   
                                        ets_bu   FOODS_1_001 3      1.034852   
                                        ets_mint FOODS_1_001 3      1.020277   
                                        ets_td   FOODS_1_001 3      1.031719   
                                        lgb_base FOODS_1_001 3      1.269884   

                                                                    ot_tdfp90  \
unique_id                               name     product     level              
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3       1.067289   
                                        ets_bu   FOODS_1_001 3       1.028848   
                                        ets_mint FOODS_1_001 3       1.146937   
                                        ets_td   FOODS_1_001 3       1.065806   
                                        lgb_base FOODS_1_001 3       1.500536   

                                                                    ot_mint90  \
unique_id                               name     product     level              
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3       1.028442   
                                        ets_bu   FOODS_1_001 3       1.027066   
                                        ets_mint FOODS_1_001 3       1.018308   
                                        ets_td   FOODS_1_001 3       1.025707   
                                        lgb_base FOODS_1_001 3       1.268725   

                                                                       ot_95  \
unique_id                               name     product     level             
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3      1.050723   
                                        ets_bu   FOODS_1_001 3      1.050723   
                                        ets_mint FOODS_1_001 3      1.034079   
                                        ets_td   FOODS_1_001 3      1.047683   
                                        lgb_base FOODS_1_001 3      1.287514   

                                                                     ot_bu95  \
unique_id                 

In [7]:

df2 = var_[np.isfinite(var_).all(axis=1)].copy()

vlist = []
product_ids = df2.index.get_level_values("product").unique().tolist()
for product_id in tqdm(product_ids):
    one = df2.xs(product_id, level="product").groupby(["level", "name"]).mean(numeric_only=True)
    vlist.append(one)

bwe_summary = pd.concat(vlist).groupby(["level", "name"]).mean(numeric_only=True)
median_bwe = df2.reset_index().drop(columns=["unique_id", "product"]).groupby(["name", "level"]).median(numeric_only=True).drop(columns=["true_demand"], errors="ignore")

bwe_summary.iloc[:, 3:].to_csv("bwe.csv")
median_bwe.reset_index().to_csv("median_bwe.csv", index=False)

print("saved:", os.path.abspath("bwe.csv"))
print("saved:", os.path.abspath("median_bwe.csv"))
bwe_summary.iloc[:, 3:]


100%|█████████████████████████████████████████████████████████████████████████████| 3042/3042 [00:05<00:00, 519.14it/s]


saved: E:\yjz\Extension for hts\JayCode\modified_inventory_files\bwe.csv
saved: E:\yjz\Extension for hts\JayCode\modified_inventory_files\median_bwe.csv


ot_bu90  ot_tdfp90  ot_mint90     ot_95    ot_bu95  \
level name                                                            
1     ets_base  6.393401   2.367998   4.085953  3.040052   9.918841   
      ets_bu    6.393401   2.231104   3.977964  2.917720   9.918841   
      ets_mint  5.751004   2.040246   3.536250  2.538616   8.833060   
      ets_td    6.501623   2.367998   4.158862  3.040052  10.026592   
      lgb_base  6.378625   2.321184   3.834550  2.786540   9.774629   
      lgb_bu    6.378625   2.253919   3.759544  2.739045   9.774629   
      lgb_mint  6.203561   2.227170   3.669818  2.663874   9.471018   
      lgb_td    6.443222   2.321184   3.884661  2.786540   9.840576   
2     ets_base  2.151857   1.394617   1.675426  1.809788   2.897420   
      ets_bu    2.151857   1.301160   1.629863  1.762125   2.897420   
      ets_mint  2.019487   1.284127   1.546957  1.656846   2.666069   
      ets_td    2.220998   1.391451   1.720307  1.844094   2.966363   
      lgb_base  2.298263   1.584495   1.780361  1.941183   3.009274   
      lgb_bu    2.298263   1.526758   1.751110  1.908882   3.009274   
      lgb_mint  2.265225   1.527479   1.739408  1.889950   2.947761   
      lgb_td    2.337130   1.579971   1.809205  1.961845   3.048575   
3     ets_base  1.223197   1.112219   1.131681  1.357996   1.357996   
      ets_bu    1.223197   1.070507   1.120304  1.357996   1.357996   
      ets_mint  1.188058   1.065382   1.094349  1.305100   1.305100   
      ets_td    1.248733   1.106733   1.149866  1.383452   1.383452   
      lgb_base  1.413386   1.303361   1.306985  1.541536   1.541536   
      lgb_bu    1.413386   1.282116   1.300243  1.541536   1.541536   
      lgb_mint  1.404386   1.273284   1.295765  1.527339   1.527339   
      lgb_td    1.414430   1.293445   1.306339  1.542698   1.542698   

                ot_tdfp95  ot_mint95     ot_99    ot_bu99  ot_tdfp99  \
level name                                                             
1     ets_base   3.040052   6.004781  4.790446  18.994158   4.790446   
      ets_bu     2.917720   5.863822  4.705291  18.994158   4.705291   
      ets_mint   2.538616   5.092579  3.847004  16.785780   3.847004   
      ets_td     3.040052   6.077507  4.790446  19.101026   4.790446   
      lgb_base   2.786540   5.452105  4.009287  18.526365   4.009287   
      lgb_bu     2.739045   5.347572  4.012970  18.526365   4.012970   
      lgb_mint   2.663874   5.175157  3.815433  17.902019   3.815433   
      lgb_td     2.786540   5.502652  4.009287  18.594846   4.009287   
2     ets_base   1.521292   2.057319  2.514842   4.836274   1.860085   
      ets_bu     1.431042   2.004250  2.473372   4.836274   1.778047   
      ets_mint   1.374391   1.849113  2.224840   4.353179   1.619696   
      ets_td     1.518123   2.102118  2.549064   4.904846   1.856908   
      lgb_base   1.666239   2.091916  2.527042   4.862182   1.889835   
      lgb_bu     1.613023   2.056835  2.497745   4.862182   1.848427   
      lgb_mint   1.604058   2.027291  2.443006   4.728738   1.814379   
      lgb_td     1.661722   2.120910  2.547825   4.902298   1.885328   
3     ets_base   1.131810   1.193094  1.715320   1.715320   1.187395   
      ets_bu     1.090557   1.180167  1.715320   1.715320   1.147348   
      ets_mint   1.079074   1.142441  1.616831   1.616831   1.119218   
      ets_td     1.126317   1.211273  1.740625   1.740625   1.181887   
      lgb_base   1.315721   1.357731  1.882571   1.882571   1.352551   
      lgb_bu     1.295254   1.350215  1.882571   1.882571   1.334165   
      lgb_mint   1.284784   1.342411  1.854962   1.854962   1.319274   
      lgb_td     1.305809   1.357185  1.883951   1.883951   1.342647   

                ot_mint99  
level name                 
1     ets_base  10.958168  
      ets_bu    10.733133  
      ets_mint   9.125364  
      ets_td    11.030553  
      lgb_base   9.639933  
      lgb_bu     9.460617  
      lgb_mint   9.080973  
      lgb_td     9.691297  
2     ets_base   

## Incoherence

In [8]:

lgb_methods = list(lgb["name"].unique())
ets_methods = list(ets["name"].unique())

df_lgb = lgb[["name", "true_demand", "ot_90", "ot_95", "ot_99"]].copy()
df_ets = ets[["name", "true_demand", "ot_90", "ot_95", "ot_99"]].copy()


methods = ["ot_90", "ot_95", "ot_99"]
prd_list = list(prd_idx.columns)


In [9]:

from typing import List


def get_direct_children(all_ids: List[str], parent_id: str) -> List[str]:
    parts = str(parent_id).split("/")
    depth = len(parts)
    children = []
    for sid in all_ids:
        sid = str(sid)
        if sid == parent_id:
            continue
        child_parts = sid.split("/")
        if len(child_parts) != depth + 1:
            continue
        if (
            child_parts[0] == parts[0]
            and child_parts[-3:] == parts[-3:]
            and child_parts[1:-3][: len(parts[1:-3])] == parts[1:-3]
        ):
            children.append(sid)
    return sorted(children)


def check_coherence_long_format(df: pd.DataFrame, value_col: str, id_col: str = "unique_id", time_col: str = "ds") -> pd.DataFrame:
    all_ids = df[id_col].astype(str).unique().tolist()
    results = []
    for parent_id in all_ids:
        children = get_direct_children(all_ids, parent_id)
        if not children:
            continue

        parent_df = df[df[id_col].astype(str) == parent_id][[time_col, value_col]].sort_values(time_col)
        children_df = df[df[id_col].astype(str).isin(children)][[time_col, id_col, value_col]]
        children_sum = children_df.groupby(time_col)[value_col].sum().reset_index()

        merged = pd.merge(parent_df, children_sum, on=time_col, suffixes=("_parent", "_children"))
        merged["diff"] = merged[f"{value_col}_parent"] - merged[f"{value_col}_children"]
        merged["abs_diff"] = merged["diff"].abs()
        merged["parent_id"] = parent_id
        results.append(merged)

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


def run_incoherence_for(df_source: pd.DataFrame, method_names: list):
    incoherence = {}
    for m in methods:
        m_inco = {}
        for fr in method_names:
            print(f"{fr} Begins", "*" * 70)
            df_r = df_source[df_source["name"] == fr].reset_index(drop=True)
            check_result = {}
            for prd in tqdm(prd_list):
                idx = prd_idx[prd].dropna().astype(int).tolist()
                if len(idx) == 0:
                    continue
                dff = pd.concat([uid.iloc[idx, :].reset_index(drop=True), df_r.iloc[idx, :].reset_index(drop=True)], axis=1)
                check_result[prd] = check_coherence_long_format(dff, m)
            m_inco[fr] = check_result
            print(f"{fr} Finished", "*" * 70)
        incoherence[m] = m_inco
    return incoherence


In [10]:

import pickle

lgb_results = run_incoherence_for(df_lgb, lgb_methods)
with open("lgb_incoherence_results.pkl", "wb") as f:
    pickle.dump(lgb_results, f)

ets_results = run_incoherence_for(df_ets, ets_methods)
with open("ets_incoherence_results.pkl", "wb") as f:
    pickle.dump(ets_results, f)

print("saved:", os.path.abspath("lgb_incoherence_results.pkl"))
print("saved:", os.path.abspath("ets_incoherence_results.pkl"))


lgb_base Begins **********************************************************************


100%|██████████████████████████████████████████████████████████████████████████████| 3049/3049 [01:20<00:00, 37.74it/s]


lgb_base Finished **********************************************************************
lgb_bu Begins **********************************************************************


100%|██████████████████████████████████████████████████████████████████████████████| 3049/3049 [01:21<00:00, 37.60it/s]


lgb_bu Finished **********************************************************************
lgb_td Begins **********************************************************************


...

ets_bu Finished **********************************************************************
ets_td Begins **********************************************************************


100%|██████████████████████████████████████████████████████████████████████████████| 3049/3049 [01:20<00:00, 37.84it/s]


ets_td Finished **********************************************************************
ets_mint Begins **********************************************************************


100%|██████████████████████████████████████████████████████████████████████████████| 3049/3049 [01:25<00:00, 35.66it/s]


ets_mint Finished **********************************************************************
saved: E:\yjz\Extension for hts\JayCode\modified_inventory_files\lgb_incoherence_results.pkl
saved: E:\yjz\Extension for hts\JayCode\modified_inventory_files\ets_incoherence_results.pkl


In [11]:

import pickle


def avg_incoherence(result_dict: dict):
    pieces = []
    for key_1 in list(result_dict.keys()):
        dict_2 = result_dict[key_1]
        dict_4 = {}
        for key_2 in tqdm(list(dict_2.keys())):
            one = dict_2[key_2]
            if len(one) == 0:
                continue
            prodcs = one.groupby("parent_id").mean(numeric_only=True)
            dict_4[key_2] = prodcs["abs_diff"] / prodcs.iloc[:, 1]
        pieces.append(pd.DataFrame({key_1: dict_4}))
    return pd.concat(pieces, axis=1) if pieces else pd.DataFrame()


lgb_incoherence = {i: avg_incoherence(lgb_results[i]) for i in list(lgb_results.keys())}
ets_incoherence = {i: avg_incoherence(ets_results[i]) for i in list(ets_results.keys())}

a = {}
for i in methods:
    a[i] = pd.concat([lgb_incoherence[i], ets_incoherence[i]], axis=1)

out_ = {}
for i in methods:
    stacked = a[i].stack(dropna=True)
    parts = []
    for (product, method), s in tqdm(stacked.items()):
        if isinstance(s, pd.Series) and len(s) > 0:
            tmp = s.rename_axis("node_name").reset_index(name="incoherence_ratio")
            tmp["product"] = product
            tmp["method"] = method
            depth = tmp["node_name"].astype(str).str.count("/")
            tmp["level"] = (depth - depth.min() + 1).astype(int)
            parts.append(tmp[["method", "product", "node_name", "level", "incoherence_ratio"]])
    out_[i] = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["method", "product", "node_name", "level", "incoherence_ratio"])

b = {}
for i in methods:
    b[i] = out_[i][["method", "level", "incoherence_ratio"]].groupby(["method", "level"]).mean(numeric_only=True)
    b[i] = b[i].rename(columns={"incoherence_ratio": f"{i}_incoherence_ratio"})

results = pd.merge(b["ot_90"], b["ot_95"], on=["method", "level"], how="outer")
results = pd.merge(results, b["ot_99"], on=["method", "level"], how="outer")
results.to_csv("incoherence_ratio_for_all_tsl.csv")

print("saved:", os.path.abspath("incoherence_ratio_for_all_tsl.csv"))
results


100%|█████████████████████████████████████████████████████████████████████████████| 3049/3049 [00:04<00:00, 687.27it/s]
C:\Users\qifei\AppData\Local\Temp\ipykernel_31400\1712161910.py:28: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = a[i].stack(dropna=True)
24392it [01:09, 353.18it/s]
C:\Users\qifei\AppData\Local\Temp\ipykernel_31400\1712161910.py:28: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = a[i].stack(dropna=True)
24392it [01:08, 353.94it/s]
C:\Users\qifei\AppData\Local\Temp\ipykernel_31400\1712161910.py:28: FutureWarning: The previous implemen

saved: E:\yjz\Extension for hts\JayCode\modified_inventory_files\incoherence_ratio_for_all_tsl.csv


ot_90_incoherence_ratio  ot_95_incoherence_ratio  \
method   level                                                     
ets_base 1                     0.044988                 0.048459   
         2                     0.068570                 0.075659   
ets_bu   1                     0.025232                 0.028748   
         2                     0.056404                 0.063602   
ets_mint 1                     0.034177                 0.037583   
         2                     0.074081                 0.080769   
ets_td   1                     0.032327                 0.035875   
         2                     0.063250                 0.070390   
lgb_base 1                     0.103163                 0.106583   
         2                     0.153922                 0.160271   
lgb_bu   1                     0.071334                 0.074665   
         2                     0.137914                 0.144301   
lgb_mint 1                     0.070461                 0.073963   
         2                     0.133829                 0.140414   
lgb_td   1                     0.073241                 0.076852   
         2                     0.131405                 0.138075   

                ot_99_incoherence_ratio  
method   level                           
ets_base 1                     0.054775  
         2                     0.088320  
ets_bu   1                     0.035164  
         2                     0.076458  
ets_mint 1                     0.043799  
         2                     0.092764  
ets_td   1                     0.042332  
         2                     0.083139  
lgb_base 1                     0.112779  
         2                     0.171588  
lgb_bu   1                     0.080690  
         2                     0.155667  
lgb_mint 1                     0.080314  
         2                     0.152140  
lgb_td   1                     0.083384  
         2                     0.149937

In [12]:
results

ot_90_incoherence_ratio  ot_95_incoherence_ratio  \
method   level                                                     
ets_base 1                     0.044988                 0.048459   
         2                     0.068570                 0.075659   
ets_bu   1                     0.025232                 0.028748   
         2                     0.056404                 0.063602   
ets_mint 1                     0.034177                 0.037583   
         2                     0.074081                 0.080769   
ets_td   1                     0.032327                 0.035875   
         2                     0.063250                 0.070390   
lgb_base 1                     0.103163                 0.106583   
         2                     0.153922                 0.160271   
lgb_bu   1                     0.071334                 0.074665   
         2                     0.137914                 0.144301   
lgb_mint 1                     0.070461                 0.073963   
         2                     0.133829                 0.140414   
lgb_td   1                     0.073241                 0.076852   
         2                     0.131405                 0.138075   

                ot_99_incoherence_ratio  
method   level                           
ets_base 1                     0.054775  
         2                     0.088320  
ets_bu   1                     0.035164  
         2                     0.076458  
ets_mint 1                     0.043799  
         2                     0.092764  
ets_td   1                     0.042332  
         2                     0.083139  
lgb_base 1                     0.112779  
         2                     0.171588  
lgb_bu   1                     0.080690  
         2                     0.155667  
lgb_mint 1                     0.080314  
         2                     0.152140  
lgb_td   1                     0.083384  
         2                     0.149937